# The 41 most strongly correlated sensor pairs

**Graph notebook 3 in `notebooks/graph/`.** Read `01_physics_fd001.ipynb` first: this notebook takes the same steps on a different graph, and uses its words — the table of zeros and ones, the rescaled table `Â`, mixing and transforming, reach.

## What this notebook is for

The paper's Conclusions say its hand-built table captures sensor relationships "beyond data-driven correlations" (its item 2), yet the paper describes no graph built from correlations for its network to check that against: it computes correlations only to choose which sensors to keep, and never says what replaced its graph in its "w/o Physics prior" row. This project builds one (assumption A-27): measure how closely each pair of sensors' readings move together on the training engines, and join the 41 pairs that move together most strongly — as many edges as the paper's graph has, so that the one difference between the two graphs is *which* pairs are joined. This notebook builds that graph step by step from seed 42's training engines, shows that every seed builds a slightly different one, follows one window through its first graph layer, and reads the stored five-seed result. (Corrected 2026-10-03: this said the paper never builds a graph from correlations to check. That is more than its text shows: it never says what replaced its graph in the "w/o Physics prior" row, its Figs. 9 and 10 are graphs whose colours and line weights come from how much each sensor moved the prediction, and its review of earlier work mentions other authors' graphs built from correlations. Found by a review that flagged the sentence as unsupported, and re-checked against the paper's text that day (assumption A-27).)

## What it does not do

- **It does not train anything.** It loads the network seed 42 trained in a stored run, and reads all five seeds' results from that run.
- **It does not compare this graph with the paper's.** `04_comparison_fd001.ipynb` does, seed by seed.
- **It does not say which relationships are physical.** A correlation says that two sensors' readings rose and fell together in these 80 engines. It says nothing about why, and the paper's graph is a claim about why.

## The notebook in one list

| # | Step | What goes in | What comes out |
|---|------|--------------|----------------|
| 1 | **Load** the stored network | the stored five-seed run of this graph | seed 42's network and its data, checked against the run |
| 2 | **Measure the correlations** | every row of seed 42's 80 training engines | how strongly each of the 91 pairs of sensors moves together, one worked by hand |
| 3 | **Keep the 41 strongest** | the 91 strengths | the table of zeros and ones the network received, and what a cut-off would have given instead |
| 4 | **Rescale it** | the same table | the table the network multiplies by: two sensors alone, and three groups it cannot tell apart |
| 5 | **Run one graph layer by hand** | one real window and the trained layer's 32 learned numbers | 14 x 32 numbers, the same as the network's own layer gives |
| 6 | **Build it for every seed** | each seed's training engines | five slightly different graphs |
| 7 | **Read the stored result** | the five seeds' scores | the mean and spread on the test engines and on the held-back engines |

## 1. Background

### How strongly two sensors move together

The **Pearson correlation** of two sensors' readings measures how closely they follow a straight line together. For readings `x` and `y` over the same `n` rows,

```
r = Σ (x_k - x̄)(y_k - ȳ)  /  sqrt( Σ (x_k - x̄)²  ×  Σ (y_k - ȳ)² )
```

- `x_k` and `y_k` are the two sensors' readings in row `k`, one row being one engine at one cycle; `x̄` and `ȳ` are their averages over the `n` rows.
- The top adds up, row by row, how far each reading is from its own average, multiplied together. It is large and positive when the two are above their averages in the same rows, large and negative when one is above where the other is below.
- The bottom divides by how far each wanders on its own, so `r` always lies between -1 and +1, whatever the units.

`r = +1` means the two rise and fall exactly together, `-1` that one rises exactly as the other falls, and 0 that no straight line relates them. With made-up readings `x = [1, 2, 3]` and `y = [2, 4, 7]`: the averages are 2 and 4.333; the distances from them are `[-1, 0, 1]` and `[-2.333, -0.333, 2.667]`; the top is `2.333 + 0 + 2.667 = 5.0`; the bottom is `sqrt(2 × 12.667) = 5.033`; so `r = 0.993`.

The graph uses the absolute value, `|r|`: a sensor that falls exactly as another rises is as closely related to it as one that rises with it. The paper uses the same Pearson correlation for a different job, to decide which sensors to keep at all (its Section 3.2.2, Equation 17).

### Why a fixed number of edges

The obvious rule, joining every pair whose `|r|` passes a cut-off, lets the number of edges float with the cut-off. Step 3 measures what it floats to on these engines. Fixing the count at the paper's own 41 leaves one difference between this graph and the paper's: which 41 pairs are joined (assumption A-27).

### Why the training engines only

The correlations decide the graph, and the graph is part of the model, so they are measured on the 80 training engines alone, after the same smoothing and rescaling the network's windows come from. Measured on all 100 engines, the 20 held back would help decide the model they are then used to judge. This is the rule the rescaling of the readings follows too (assumption A-25). Because the seed decides which 20 engines are held back, each seed measures its correlations on different rows and can build a different graph; step 6 shows how different.

## 2. Setup

The imports, then the stored runs, chosen by the rule `notebooks/graph/README.md` gives: the newest commit at which all three graphs have a stored five-seed run of `config/fd001.yaml` that saved its trained networks, and the newest run of each graph at that commit. Every notebook in this folder chooses the same way, so all four read the same runs. If a run is missing, the cell stops and prints the commands that make them.

In [ ]:
from __future__ import annotations

import logging
import sys
from dataclasses import replace
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

# The project's top folder is the nearest one above this notebook that holds
# pyproject.toml. Adding its src/ folder to Python's search path makes the
# package importable from a clone that was never installed.
PROJECT_ROOT = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "pyproject.toml").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

try:
    import torch
except ImportError as missing:
    raise ImportError(
        "This notebook loads a trained network, which needs PyTorch, and PyTorch is "
        'not installed. Install the project with  pip install -e ".[model]"'
    ) from missing

from thesis_gnn_rul_xai.config import load_config
from thesis_gnn_rul_xai.data.pipeline import run_preprocessing
from thesis_gnn_rul_xai.data.sensors import PAPER_SENSOR_COLUMNS, UNIT_COL
from thesis_gnn_rul_xai.graph.adjacency import (
    adjacency_report,
    adjacency_to_frame,
    build_correlation_adjacency,
    build_physics_adjacency,
    connections_of,
    normalize_adjacency,
    to_networkx,
)
from thesis_gnn_rul_xai.models import ModelConfig
from thesis_gnn_rul_xai.training import TrainingConfig, evaluate_model, find_runs, load_seed
from thesis_gnn_rul_xai.utils.logging_setup import configure_logging

configure_logging(logging.ERROR)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch     : {torch.__version__}")

In [ ]:
config = load_config(PROJECT_ROOT / "config" / "fd001.yaml")
GRAPHS = ("physics", "complete", "correlation")
GRAPH = "correlation"    # the graph this notebook shows
SEED = 42              # the network every worked example uses
COMMAND = "python scripts/train.py --config config/fd001.yaml --graph {} --seeds 5"
# The training settings a run records in its summary.json, at their default values.
TRAINING = {key: getattr(TrainingConfig(), key) for key in (
    "epochs", "batch_size", "learning_rate", "final_learning_rate", "early_stopping_patience")}


def stored_runs(kind: str) -> list:
    # Five seeds of config/fd001.yaml with only the graph changed, the trained
    # networks saved, the network sizes of the paper's Table 4, and the default
    # training settings.
    return [
        stored for stored in find_runs(
            preprocessing=replace(config, graph=kind), seeds=5, with_weights=True,
            split="varies_with_seed", early_stopping=True, lr_schedule="step",
            temporal_stage="last_cycle",
        )
        if stored.model_config == ModelConfig()
        and {key: stored.summary.get("training", {}).get(key) for key in TRAINING} == TRAINING
    ]


found = {kind: stored_runs(kind) for kind in GRAPHS}
shared = set.intersection(*({stored.summary["commit"] for stored in runs} for runs in found.values()))
if not shared:
    missing = [kind for kind, runs in found.items() if not runs]
    raise FileNotFoundError(
        ("reports/ holds no stored five-seed run, with its networks saved, for: "
         + ", ".join(missing) if missing
         else "The stored runs of the three graphs come from different commits.")
        + "\nFrom one committed state of the code, run\n\n"
        + "\n".join("    " + COMMAND.format(kind) for kind in GRAPHS)
        + "\n\nand run this notebook again."
    )

# The newest commit at which all three graphs were stored, and the newest run
# of each graph at that commit.
newest = {
    commit: max(stored.summary["finished_utc"] for runs in found.values()
                for stored in runs if stored.summary["commit"] == commit)
    for commit in shared
}
commit = max(shared, key=newest.get)
runs = {kind: [s for s in found[kind] if s.summary["commit"] == commit][-1] for kind in GRAPHS}
run = runs[GRAPH]

print(f"Commit of all three graphs' runs: {commit}")
for kind, stored in runs.items():
    print(f"  {kind:12s} {stored.name}")

## 3. The trained network

### Step 1 — Load the stored network

**What.** Load the network that seed 42 trained in the run chosen above, from the `model.pt` saved beside it, and rebuild its data with the same seed.

**Why.** The worked examples below should show the graph and the layer a network of this project's reported runs actually used. A network trained here instead would be one seed and would record no commit.

**Check.** The loaded network must score the test engines exactly as the run recorded for seed 42, and the graph inside it must be the rescaled table the rebuilt data holds.

In [ ]:
model, prepared = load_seed(run, SEED)
sensors = list(prepared.sensor_symbols)

recorded = float(run.results.set_index("seed").loc[SEED, "rmse"])
scored = evaluate_model(model, prepared.test)["rmse"]
same_network = bool(np.isclose(scored, recorded, rtol=0, atol=1e-6))
same_graph = bool(np.allclose(model.adjacency.numpy(), prepared.adjacency_normalized))
assert same_network and same_graph
print(f"OK  test RMSE {scored:.6f} cycles, as the run recorded for seed {SEED}.")
print("OK  The network's graph is the rescaled table of the rebuilt data.")

## 4. The graph, step by step

### Step 2 — Measure the correlations

**What.** Rebuild seed 42's data once more, this time keeping the table of rows the windows are cut from, take the rows of the 80 training engines, and compute `|r|` for every pair of the 14 sensors. Then work one pair by hand: the pair with the largest `|r|`.

**Why.** These 91 numbers are all the graph is built from.

**Check.** The rows must come from exactly the network's 80 training engines, and the hand computation must give the same `r` as pandas.

In [ ]:
rebuilt = run_preprocessing(replace(run.preprocessing, seed=SEED), keep_frames=True)
rows = rebuilt.frames["train"]                      # every engine of the training file, smoothed and rescaled
training_engines = np.unique(prepared.train.units)
same_split = bool(np.array_equal(np.unique(rebuilt.train.units), training_engines))
training_rows = rows[rows[UNIT_COL].isin(training_engines)][list(PAPER_SENSOR_COLUMNS)]
training_rows.columns = sensors                     # a DataFrame, one column per sensor
del rebuilt, rows
assert same_split
print(f"{len(training_rows)} rows from {len(training_engines)} training engines, "
      f"{training_rows.shape[1]} sensors.\n")

correlation = training_rows.corr(method="pearson")  # DataFrame, shape (14, 14)
strength = correlation.abs()
display(strength.round(2))

upper_i, upper_j = np.triu_indices(len(sensors), k=1)         # the 91 pairs, each once
pair_strength = strength.to_numpy()[upper_i, upper_j]
order = np.argsort(-pair_strength, kind="stable")             # strongest first; ties in sensor order
a, b = sensors[upper_i[order[0]]], sensors[upper_j[order[0]]]
xa, xb = training_rows[a].to_numpy(float), training_rows[b].to_numpy(float)
top = ((xa - xa.mean()) * (xb - xb.mean())).sum()
spread_a, spread_b = ((xa - xa.mean()) ** 2).sum(), ((xb - xb.mean()) ** 2).sum()
r_by_hand = top / np.sqrt(spread_a * spread_b)
print(f"The strongest pair is {a} and {b}, over n = {len(xa)} rows:")
print(f"  top    = Σ (x - x̄)(y - ȳ)            = {top:.2f}")
print(f"  bottom = sqrt({spread_a:.2f} × {spread_b:.2f}) = {np.sqrt(spread_a * spread_b):.2f}")
print(f"  r      = {top:.2f} / {np.sqrt(spread_a * spread_b):.2f} = {r_by_hand:+.4f}"
      f"     pandas gives {correlation.loc[a, b]:+.4f}")
r_matches = bool(np.isclose(r_by_hand, correlation.loc[a, b], atol=1e-9))
assert r_matches

Two things in that output repay a second look.

- **The strongest pair moves in opposite directions.** `r` for `Ps30` and `phi` is negative: where one is above its average, the other tends to be below. Its absolute value, 0.94, is what ranks it first.
- **Both sums of squares equal the number of rows.** The readings were rescaled on exactly these 80 training engines so that each sensor averages 0 with a spread of 1 (assumption A-03), and that makes `Σ (x - x̄)²` equal to `n` for every sensor. The formula then reduces to the average of the products, `r = Σ x_k y_k / n`. On any other rows, such as all 100 engines, that shortcut would not hold, and the full formula is what pandas computes.

### Step 3 — Keep the 41 strongest

**What.** Rank the 91 pairs by `|r|`, strongest first, join the first 41, and compare the result with the paper's graph. Then, for contrast, build the graph a cut-off would have given.

**Why.** This is the whole of the rule (assumption A-27), and the contrast shows why it is a fixed count and not a cut-off.

**Check.** The table built here must be exactly the one inside the stored network's data, and hold 41 edges.

In [ ]:
raw = prepared.adjacency_raw                        # NumPy array, shape (14, 14)
rebuilt_graph, _ = build_correlation_adjacency(training_rows, sensors, n_edges=41)
same_as_network = bool((rebuilt_graph == raw).all())
report = adjacency_report(raw, sensors)
assert same_as_network and report["n_edges"] == 41
print(f"OK  The 41 strongest pairs are exactly the network's graph: {report['n_edges']} edges.\n")
display(adjacency_to_frame(raw.astype(int), sensors))

physics = build_physics_adjacency()[0]
in_table_3 = physics[upper_i, upper_j] == 1
kept = raw[upper_i, upper_j] == 1
shared_with_table_3 = int((kept & in_table_3).sum())
weakest_kept = pair_strength[kept].min()
strongest_left_out = pair_strength[~kept].max()
print(f"Weakest pair kept   |r| = {weakest_kept:.4f}")
print(f"Strongest left out  |r| = {strongest_left_out:.4f}")
print(f"Edges shared with Table 3: {shared_with_table_3} of 41")
print(f"Sensors joined to nothing: {report['isolated_nodes']}")
for alone in report["isolated_nodes"]:
    others = strength[alone].drop(alone)
    print(f"  {alone}'s strongest correlation with any sensor: {others.max():.3f}, with {others.idxmax()}")

around_the_cut = pd.DataFrame([
    {"rank": rank + 1, "pair": f"{sensors[upper_i[k]]}-{sensors[upper_j[k]]}",
     "|r|": round(float(pair_strength[k]), 4), "kept": rank < 41, "in Table 3": bool(in_table_3[k])}
    for rank, k in enumerate(order)
][36:46]).set_index("rank")
print("\nRanks 37 to 46, around the cut:")
display(around_the_cut)

In [ ]:
# All 91 pairs, strongest first. A tick under a bar marks a pair the paper's graph also joins.
ranked = pair_strength[order]
figure, axis = plt.subplots(figsize=(10, 3.6))
axis.bar(np.arange(41), ranked[:41], width=0.8, color="#2a78d6", label="kept: the 41 strongest pairs")
axis.bar(np.arange(41, 91), ranked[41:], width=0.8, color="#c3c2b7", label="left out")
marks = np.flatnonzero(in_table_3[order])
axis.scatter(marks, np.full(len(marks), -0.04), marker="|", s=40, color="#0b0b0b",
             label="also joined in Table 3")
axis.axhline(weakest_kept, color="#52514e", linewidth=0.8, linestyle="--")
axis.text(90, weakest_kept + 0.02, f"weakest kept, {weakest_kept:.3f}", ha="right", color="#52514e")
axis.set_xlim(-1, 91)
axis.set_ylim(-0.08, 1.02)
axis.set_xlabel("the 91 pairs of sensors, strongest first")
axis.set_ylabel("|r| on the training engines")
axis.set_title("Which pairs the correlation graph joins", loc="left")
axis.grid(axis="y", color="#e6e5e1", linewidth=0.8)
axis.spines[["top", "right"]].set_visible(False)
axis.legend(frameon=False, loc="center right", bbox_to_anchor=(1, 0.58))
plt.tight_layout()
plt.show()

The cut falls inside a smooth run of values: the weakest pair kept and the strongest left out differ in the third decimal place, so which side of the cut a pair lands on is close to arbitrary for the pairs near it. Most of the pairs the paper joins rank below the cut, and the two sensors joined to nothing correlate with no other sensor anywhere near as strongly as the 41 kept pairs do. `P15`'s raw readings take only two distinct values in the whole training file (finding F-01).

**What a cut-off would have given.** The next cell builds the graph from every pair whose `|r|` reaches a cut-off, for the cut-offs tried when the rule was chosen (assumption A-27), and counts what each graph holds. "Largest group" is the largest set of sensors that can all reach each other along edges; "all joined" says whether every pair inside it is joined directly.

In [ ]:
cut_offs = []
for cut_off in (0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95):
    graph, _ = build_correlation_adjacency(training_rows, sensors, threshold=cut_off)
    network = to_networkx(graph, sensors)
    largest = max(nx.connected_components(network), key=len)
    size = len(largest)
    cut_offs.append({
        "cut-off": cut_off,
        "edges": int(graph.sum() // 2),
        "joined to nothing": ", ".join(s for s in sensors if network.degree(s) == 0) or "none",
        "largest group": size if size > 1 else 0,
        "all joined": size > 1 and network.subgraph(largest).number_of_edges() == size * (size - 1) // 2,
    })
display(pd.DataFrame(cut_offs).set_index("cut-off"))

From 0.5 to 0.7 a cut-off joins twelve sensors, every one to every other. Once each row is divided by its total, those twelve rows are identical, and all twelve sensors would take the same average — the flattening `02_complete_fd001.ipynb` showed for all fourteen. A graph like that would test how many edges there are, not which. Fixing the count at the paper's 41 avoids it.

### Step 4 — Rescale it

**What.** Add 1 down the diagonal and divide every row by its total, as for the other graphs.

**Why.** It is the table the network multiplies by, and two of its rows are unlike anything in the paper's graph.

**Check.** Every row must add up to 1, the table must be what `normalize_adjacency(raw, method="row")` gives, and the rows of the two sensors joined to nothing must hold a single 1, on the diagonal.

In [ ]:
a_hat = prepared.adjacency_normalized          # NumPy array, shape (14, 14)
display(adjacency_to_frame(a_hat, sensors).round(3))
rows_sum_to_one = bool(np.allclose(a_hat.sum(axis=1), 1.0, atol=1e-6))
matches_function = bool(np.allclose(a_hat, normalize_adjacency(raw, method="row")))
alone = [sensors.index(s) for s in report["isolated_nodes"]]
alone_keep_themselves = all(np.isclose(a_hat[i, i], 1.0) and np.isclose(a_hat[i].sum(), 1.0) for i in alone)
assert rows_sum_to_one and matches_function and alone_keep_themselves


def identical_rows(matrix: np.ndarray, names: list[str]) -> list[list[str]]:
    groups: dict[bytes, list[str]] = {}
    for name, row in zip(names, np.round(np.asarray(matrix, dtype=float), 9)):
        groups.setdefault(row.tobytes(), []).append(name)
    return [group for group in groups.values() if len(group) > 1]


print("Groups of sensors with identical rows: "
      + "; ".join(", ".join(group) for group in identical_rows(a_hat, sensors)))
n_different_rows = len(np.unique(np.round(a_hat.astype(float), 9), axis=0))
print(f"{n_different_rows} different rows among the 14.")
pieces = sorted(nx.connected_components(to_networkx(raw, sensors)), key=len, reverse=True)
print("The graph falls into " + str(len(pieces)) + " pieces: "
      + " | ".join(", ".join(sorted(piece)) for piece in pieces))
main = to_networkx(raw, sensors).subgraph(pieces[0])
print(f"Within the largest piece, the furthest two sensors are {nx.diameter(main)} edges apart.")

Three things differ from the paper's graph here.

- **Two sensors never mix.** The rows of the two sensors joined to nothing hold a single 1, on the diagonal, so their average is their own reading, in every graph layer. Whatever the network learns about them, it learns from each alone.
- **The graph is in pieces.** Everything joined sits in one piece, and in that piece no two sensors are more than two edges apart, so three graph layers reach every pair in it. Between pieces nothing passes at all, however many layers there are; the paper's graph is in one piece, with four pairs out of reach of three layers.
- **Different groups are indistinguishable.** Three groups have identical rows, as in the paper's graph, but not the same three, and the network is left with 10 different rows instead of 9.

### Step 5 — One graph layer, by hand

**What.** The same window `01_physics_fd001.ipynb` followed, the first test engine's, through this network's first graph layer, with this network's own learned `W(1)`.

**Why.** To see a sensor joined to nothing pass through the mixing unchanged.

**Check.** The hand computation must give the layer's 14 x 32 numbers, and the mixing must leave the two lone sensors' readings as they were.

In [ ]:
window = prepared.test.features[0]                 # NumPy array, shape (14, 30)
x = window[:, -1]                                  # shape (14,): each sensor's last reading
mixed = a_hat @ x                                  # shape (14,)
print(f"Test engine {prepared.test.units[0]}, window ending at cycle {prepared.test.end_cycles[0]}\n")
print(pd.DataFrame({"X, last reading": x, "Â X, after mixing": mixed}, index=sensors).round(4).T.to_string())
lone_unchanged = bool(np.allclose(mixed[alone], x[alone]))

w1 = model.graph_layers[0].weight.detach().numpy()   # shape (1, 32)
by_hand = np.maximum(mixed[:, None] * w1, 0.0)        # shape (14, 32)
with torch.no_grad():
    carried = model.temporal(torch.from_numpy(window[None]))
    by_layer = torch.relu(model.graph_layers[0](carried, model.adjacency))[0].numpy()
layer_matches = bool(np.abs(by_hand - by_layer).max() < 1e-6)
different_after_layer_1 = len(np.unique(np.round(by_layer.astype(float), 9), axis=0))
assert lone_unchanged and layer_matches
print(f"\nOK  {' and '.join(report['isolated_nodes'])} leave the mixing exactly as they came in.")
print(f"OK  The hand computation gives the layer's {by_layer.shape[0]} x {by_layer.shape[1]} numbers "
      f"(largest difference {np.abs(by_hand - by_layer).max():.1e}); "
      f"{different_after_layer_1} different rows leave it.")

### Step 6 — Build it for every seed

**What.** Rebuild the data for each of the run's five seeds and record the graph each builds: its weakest kept and strongest left-out pair, the edges it shares with Table 3, the sensors it joins to nothing, how many different rows its rescaled table has, and how many of its edges differ from seed 42's graph.

**Why.** The seed decides which 20 engines are held back, so each seed's network was trained on its own graph. The five-seed result of step 7 averages over five slightly different graphs, and it is worth knowing how different.

**Check.** Every seed's rebuilt graph must have 41 edges.

In [ ]:
by_seed, edge_sets = [], {}
for seed in run.seeds:
    data = run_preprocessing(replace(run.preprocessing, seed=seed), keep_frames=True)
    frame = data.frames["train"]
    rows_of_seed = frame[frame[UNIT_COL].isin(np.unique(data.train.units))][list(PAPER_SENSOR_COLUMNS)]
    values = rows_of_seed.corr(method="pearson").abs().to_numpy()[upper_i, upper_j]
    graph = data.adjacency_raw
    kept_here = graph[upper_i, upper_j] == 1
    edge_sets[seed] = set(connections_of(graph))
    by_seed.append({
        "seed": seed,
        "edges": int(graph.sum() // 2),
        "weakest kept": round(float(values[kept_here].min()), 4),
        "strongest left out": round(float(values[~kept_here].max()), 4),
        "shared with Table 3": int((kept_here & in_table_3).sum()),
        "joined to nothing": ", ".join(s for s, d in zip(sensors, graph.sum(axis=1)) if d == 0),
        "different rows": len(np.unique(np.round(data.adjacency_normalized.astype(float), 9), axis=0)),
    })
    del data, frame, rows_of_seed

for row in by_seed:
    row["edges not in seed 42's graph"] = len(edge_sets[row["seed"]] - edge_sets[SEED])
seed_table = pd.DataFrame(by_seed).set_index("seed")
display(seed_table)
every_seed_has_41 = bool((seed_table["edges"] == 41).all())
in_every_graph = set.intersection(*edge_sets.values())
assert every_seed_has_41
print(f"{len(in_every_graph)} of the 41 edges are in all five seeds' graphs.")

The five graphs are close: at most two edges differ from seed 42's, and the same two sensors are joined to nothing in every one. On every seed the cut falls between 0.845 and 0.855. The step 7 result is therefore the result of five nearly identical graphs, not of one.

## 5. The stored result

### Step 7 — Read the stored result

**What.** The scores of all five seeds of the run, on the four measures notebook 02's step 11 defines, and the error on each seed's held-back engines.

**Why.** One seed is not a result (working rule 3).

**Check.** The means and spreads computed here must agree with the run's own `summary.json`.

In [ ]:
results = run.results.set_index("seed")
columns = {
    "rmse": "test RMSE", "score": "test NASA score", "r2": "test R-squared",
    "asue": "test ASUE", "validation_rmse": "held-back RMSE", "best_epoch": "pass kept",
}
per_seed = results[list(columns)].rename(columns=columns)
measures = list(columns.values())[:5]
shown = per_seed.astype(object)
for m in measures:
    shown[m] = [f"{value:.3f}" for value in per_seed[m]]
shown.loc["mean ± standard deviation"] = (
    [f"{per_seed[m].mean():.3f} ± {per_seed[m].std(ddof=1):.3f}" for m in measures] + [""]
)
print(f"Run    : {run.name}")
print(f"Commit : {run.summary['commit']}, finished {run.summary['finished_utc']} (UTC)\n")
display(shown)

# The run wrote its own mean and spread into summary.json; they must agree.
matches_summary = all(
    np.isclose(results[m].mean(), run.summary["mean"][m])
    and np.isclose(results[m].std(ddof=1), run.summary["std"][m])
    for m in ("rmse", "score", "r2", "asue")
)
assert matches_summary
print("OK  The means and spreads agree with the run's own summary.json.")

**No published figure matches this configuration.** The paper describes no graph built from correlations for its network, and never says what its "w/o Physics prior" row used in place of its graph (assumption A-19), so no published figure can be set beside these numbers. (Corrected 2026-10-03: this said the paper never builds a graph from correlations, so it reports nothing to set these numbers beside. It describes none for its network, but it never says what replaced its graph in that row, so whether the row used one cannot be told from it. Found by a review that flagged the sentence as unsupported.) `04_comparison_fd001.ipynb` sets them beside the paper's graph under exactly the same configuration, seed by seed.

## 6. Final verification

Every check in this notebook, in one place.

In [ ]:
checks = [
    ("The loaded network scores what the run recorded", same_network, f"seed {SEED}"),
    ("The network's graph is the rebuilt data's", same_graph, run.summary["commit"]),
    ("Correlations use the network's training engines", same_split,
     f"{len(training_engines)} engines"),
    ("r by hand equals pandas", r_matches, f"{a} and {b}"),
    ("The 41 strongest pairs are the network's graph", same_as_network, "41 edges"),
    ("Every row of the rescaled table adds up to 1", rows_sum_to_one, "row rescaling"),
    ("The rescaled table is normalize_adjacency's", matches_function, 'method="row"'),
    ("Sensors joined to nothing keep only themselves", alone_keep_themselves,
     ", ".join(report["isolated_nodes"])),
    ("Mixing leaves them unchanged", lone_unchanged, "test engine 1"),
    ("One graph layer by hand equals the network's", layer_matches,
     f"{different_after_layer_1} different rows out"),
    ("Every seed's graph has 41 edges", every_seed_has_41, f"seeds {run.seeds}"),
    ("Five-seed means agree with summary.json", matches_summary, run.name[:40] + "..."),
]

width = max(len(name) for name, _, _ in checks) + 2
print(f"{'CHECK':<{width}} RESULT   DETAIL")
print("-" * (width + 40))
for name, passed, detail in checks:
    print(f"{name:<{width}} {'PASS' if passed else 'FAIL':<8} {detail}")
print("-" * (width + 40))
failed = [name for name, passed, _ in checks if not passed]
if failed:
    raise AssertionError(f"{len(failed)} check(s) failed: {failed}")
print(f"All {len(checks)} checks passed.")

## 7. Summary

### What this notebook produced

- **The correlation graph, built step by step** from seed 42's 80 training engines: 91 strengths, one of them worked by hand, the 41 strongest kept, and the result checked against the graph inside the stored network.
- **What it holds**: fewer than half of its edges are ones the paper also draws, two sensors are joined to nothing, and the graph falls into pieces.
- **Why it is a fixed count and not a cut-off**: on these engines a cut-off between 0.5 and 0.7 joins twelve sensors every one to every other.
- **Five seeds, five slightly different graphs**, and the stored five-seed result of the networks trained on them, with the commit that produced it.

### Decisions that were not read from the paper

| Decision | Value used | Written up as |
|---|---|---|
| Building a graph from the data at all | The 41 most strongly correlated pairs, measured on the training engines, ties to the pair first in sensor order | A-27 |
| Which rows the correlations are measured on | The training engines' rows, after smoothing and rescaling | A-25 |
| How the table is rescaled | Each row of `A + I` divided by its own total | A-21 |
| What a sensor carries into the graph layers | The last reading of its window, one number | A-07 |
| Held-back engines and the parameters kept | 20 % of the training engines; the parameters of the pass that did best on them | A-06 |

### References

- Wen, J.-R., et al. (2026). Explainable spatiotemporal graph neutral network with dynamic SHAP enhancement for aeroengine RUL prediction. *Aerospace Science and Technology*, 168, 110875.

Full citations are in `docs/REFERENCES.md`.